In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
from fraud_detection.config import load_config
from fraud_detection.data.load import PROJECT_ROOT, load_interim
from fraud_detection.data.split import assign_split
from fraud_detection.evaluation.metrics import evaluate
from fraud_detection.features.preprocessing import TARGET, Preprocessor
from fraud_detection.imbalance.resampling import product_smote
from fraud_detection.models.xgb import train_from_config

In [ ]:
data_cfg = load_config("data")
df = load_interim()
split = assign_split(df, **data_cfg["split"])
train, val = df[split == "train"], df[split == "val"]

In [ ]:
prep = Preprocessor(**data_cfg["preprocessing"]).fit(train)
X_train, y_train = prep.transform(train), train[TARGET]
X_val, y_val = prep.transform(val), val[TARGET]

## Pondération des fraudes avec XGB

On indique ici à XGB que chaque fraude compte autant que les transactions légitimes réunies.

In [ ]:
cfg = load_config("xgboost_tuned")

In [ ]:
ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight = {ratio:.1f}")

In [ ]:
cfg["name"] = "xgb_weighted"
cfg["params"]["scale_pos_weight"] = float(ratio)
model_w, params_w, _ = train_from_config(X_train, y_train, X_val, y_val, cfg)

In [ ]:
scores_w = model_w.predict_proba(X_val)[:, 1]
pd.Series(evaluate(y_val, scores_w)).round(4)

## SMOTE par produit

Fraudes synthétiques générées séparément au sein de chaque produit, sans imputation : les valeurs manquantes structurelles et les catégories sont préservées.

In [ ]:
imb_cfg = load_config("imbalance")
X_res, y_res = product_smote(X_train, y_train, **imb_cfg["product_smote"])

In [ ]:
pd.crosstab(X_res["ProductCD"], y_res)

In [ ]:
def by_product(d):
    return d.groupby("ProductCD", observed=True)

In [ ]:
synth = X_res.iloc[len(X_train) :]
real = X_train[y_train == 1]

pd.DataFrame(
    {
        "montant médian (réel)": by_product(real)["TransactionAmt"].median(),
        "montant médian (synth.)": by_product(synth)["TransactionAmt"].median(),
        "V257 manquante (réel)": by_product(real)["V257"].apply(
            lambda s: s.isna().mean()
        ),
        "V257 manquante (synth.)": by_product(synth)["V257"].apply(
            lambda s: s.isna().mean()
        ),
    }
).round(3)

In [ ]:
cfg = load_config("xgboost_tuned")
cfg["name"] = "xgb_product_smote"
model_ps, _, _ = train_from_config(X_res, y_res, X_val, y_val, cfg)

scores_ps = model_ps.predict_proba(X_val)[:, 1]
pd.Series(evaluate(y_val, scores_ps)).round(4)

## Comparaison

In [ ]:
cfg = load_config("xgboost_tuned")
model_ref, _, _ = train_from_config(X_train, y_train, X_val, y_val, cfg)
scores_ref = model_ref.predict_proba(X_val)[:, 1]

results = pd.DataFrame(
    {
        "Sans traitement": evaluate(y_val, scores_ref),
        "Pondération": evaluate(y_val, scores_w),
        "SMOTE par produit": evaluate(y_val, scores_ps),
    }
).T.round(4)
results

Ni la pondération des classes ni un SMOTE adapté à la structure par produit n’améliorent la PR-AUC (0,608 sans traitement, contre 0,598 et 0,588). Le rééquilibrage améliore en revanche la précision sur les 1% de transactions les plus suspectes (jusqu’à 94,1% contre 92,4%), ce qui peut être pertinent pour une équipe qui souhaite avant tout traiter ces cas spécifiques. À 2%, qui est le cas le plus souvent étudié, une simple pondération est meilleure. Cependant, ces écarts peuvent aussi s'expliquer par l'aléatoire, c'est pour cela que nous allons réaliser des tests sur différents random_state.

## Comparaisons sur différents random_state

In [ ]:
def run_variant(X, y, seed, extra_params=None):
    """Entraîne XGBoost (paramètres optimisés) avec une graine donnée et évalue sur la validation."""
    cfg = load_config("xgboost_tuned")
    cfg["params"]["random_state"] = seed
    cfg["params"].update(extra_params or {})
    model, _, _ = train_from_config(X, y, X_val, y_val, cfg)
    return evaluate(y_val, model.predict_proba(X_val)[:, 1])

In [ ]:
SEEDS = [0, 1, 2]

rows = []
for seed in SEEDS:
    print(f"Seed {seed}")
    rows.append(
        {
            "variante": "Sans traitement",
            "seed": seed,
            **run_variant(X_train, y_train, seed),
        }
    )
    rows.append(
        {
            "variante": "Pondération",
            "seed": seed,
            **run_variant(X_train, y_train, seed, {"scale_pos_weight": float(ratio)}),
        }
    )

    smote_params = {**imb_cfg["product_smote"], "random_state": seed}

    X_r, y_r = product_smote(X_train, y_train, **smote_params)
    rows.append(
        {"variante": "SMOTE par produit", "seed": seed, **run_variant(X_r, y_r, seed)}
    )

    X_r, y_r = product_smote(
        X_train, y_train, **{**smote_params, "sampling_strategy": 0.1}
    )
    rows.append(
        {
            "variante": "SMOTE par produit (10%)",
            "seed": seed,
            **run_variant(X_r, y_r, seed),
        }
    )

runs = pd.DataFrame(rows)

In [ ]:
key_metrics = ["pr_auc", "precision@1%", "precision@2%", "recall@2%", "recall@5%"]
summary = runs.groupby("variante")[key_metrics].agg(
    lambda s: f"{s.mean():.4f} ± {s.std():.4f}"
)
summary

In [ ]:
runs.to_csv(PROJECT_ROOT / "reports" / "imbalance_runs.csv", index=False)

XGBoost avec les paramètres optimisés, moyenne ± écart-type sur 3 graines
(résultats détaillés : `imbalance_runs.csv`).

| Variante | PR-AUC | Précision@1% | Précision@2% | Rappel@2% | Rappel@5% |
|---|---|---|---|---|---|
| Sans traitement | 0,605 ± 0,003 | 0,924 ± 0,001 | 0,763 ± 0,002 | 0,450 ± 0,001 | 0,644 ± 0,004 |
| Pondération | 0,592 ± 0,005 | 0,930 ± 0,000 | 0,768 ± 0,005 | 0,453 ± 0,003 | 0,624 ± 0,006 |
| SMOTE par produit (0,25) | 0,586 ± 0,001 | 0,934 ± 0,007 | 0,748 ± 0,006 | 0,441 ± 0,004 | 0,621 ± 0,005 |
| SMOTE par produit (0,1) | 0,604 ± 0,005 | 0,930 ± 0,005 | 0,766 ± 0,006 | 0,452 ± 0,003 | 0,638 ± 0,007 |


## Conclusion

Aucun traitement du déséquilibre n'améliore la PR-AUC. Un SMOTE léger (ratio 0,1) fait jeu égal avec l'absence de traitement, et une pondération et un SMOTE plus fort (ratio 0,25) la dégradent nettement. Le rééquilibrage apporte seulement un léger gain de précision sur les 1 % de transactions les plus suspectes. Dans ce cas précis, il peut être intéressant de procéder au SMOTE si les usecase le requièrent.